In [1]:
import pickle
import os
from IPython.core.display import display, HTML
import pandas as pd
import sys
import re

In [2]:
table_dir = '../data/'
ds_train_data = pickle.load(open(os.path.join(table_dir, 'all_train_data_pre.pkl'), 'rb'))

In [3]:
# print(len(ds_train_data))
# print(type(ds_train_data))
# print(ds_train_data[0])

In [4]:
val_test_dict = {c['pii'] + '__' + str(c['t_idx']) : c for c in ds_train_data}
val_test_keys = list(val_test_dict.keys())

In [5]:
def get_json_dict(pii_tid):
    print(val_test_dict[pii_tid].keys())
    return val_test_dict[pii_tid]
        

def show_comp_tables(pii_tid):
    d = get_json_dict(pii_tid)
    display(HTML(f'<a href="https://www.sciencedirect.com/science/article/pii/{d["pii"]}", target="_blank">Paper Link</a>'))
    print(f'Pii, tid in paper (0-based): {d["pii"]}, {d["t_idx"]}')
    table = d['act_table'].copy()
    #display(pd.DataFrame(table))
    r, c = d['num_rows'], d['num_cols']
    if not 'row_label' in d:
        print(d['pii'], d['t_idx'])
    for i in range(r):
        # table[i] = [(d['row_label'][i], d['row_nom_exp'][i])] + table[i]
        table[i] = [d['row_label'][i]] + table[i]
    table = [['']] + table
    for i in range(c):
        # table[0].append((d['col_label'][i], d['col_nom_exp'][i]))
        table[0].append(d['col_label'][i])
    print(f'comp_table: {d["comp_table"]}')
    print(f'Prop table: {d["prop_table"]}')
    #print(f'row_labels {d["row_label"]}')
    #print(f'col_labels {d["col_label"]}')
    print(f'gid_row_labels {d["gid_row_label"]}')
    print(f'gid_col_labels {d["gid_col_label"]}')
    print()
    print(d['caption'].replace('\n', ' '))
    display(pd.DataFrame(table))
    print()
    #print(d['prop_tuples'])

In [6]:
prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1020, 1011, 1118, 70, 1306]
prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'SofP', 'BulkModulus', 'ActivationEnergy']
my_dict = {prop_names[i]: 0 for i in range(20)}
pii_tid_ds = {prop_names[i]: set() for i in range(20)}
all_rows, all_cols = [], []
pii_tidx_ds = set()
for table in ds_train_data:
    #print(table.keys())
    all_rows += table['row_label']
    all_cols += table['col_label']
    tab_row_col = table['row_label'] + table['col_label']
    pii_tidx_ds.add((table['pii'], table['t_idx']))
    for ind, elem in enumerate(tab_row_col):
        if elem>=4:
            pii_tid_ds[prop_names[elem-4]].add((table['pii'], table['t_idx'], ind))
# print('No of rows detected for each prop:')
for keys in range(4, 24):
    my_dict[prop_names[keys-4]] = all_rows.count(keys) + all_cols.count(keys)

In [7]:
pickle.dump(pii_tid_ds, open(os.path.join(table_dir, 'check_precision_ds_train.pkl'), 'wb'))

In [8]:
table_dir = '../data/'
train_data = pickle.load(open(os.path.join(table_dir, 'train_data.pkl'), 'rb'))
print(len(train_data))

4408


In [9]:
train_dict = {(c['pii'], c['t_idx']) : c for c in train_data}
train_keys = set(train_dict.keys())
print(len(train_keys))
#train_dict[('S0022309300001691', 1)]

4408


In [10]:
# train_dict[('S0022309303003624', 0)]

In [11]:
leftover_keys = train_keys - pii_tidx_ds
# leftover_keys = train_keys
print(len(leftover_keys))
keys_to_extract  = list(leftover_keys)

3983


In [12]:
subs_dict = {'Density':['density','Density','rho','r', 'd', 'D', 'Density d', 'Density, d', 'Approx. density', 'Density r', 'Density, r', 'Bulk Density', 'Bulk density', 'bulk density', 'Bulk density d', 'Bulk density r'],
'GlassTransitionTg':['Tg','T g', 'T  g', 'Tg/degC', 'Tg/K', 'glass transition','glass transition temperature', 'Glass Transition','Glass Transition Temperature', 'Glass transition','Glass transition temperature', 'Glass transitiontemperature', 'T g-onset', 'T g onset', 'Tg onset'],
'CrystallizationTemp':['Tx', 'T x', 'T  x', 'T X', 'T  X', 'T  x', 'crystallisation temperature','crystallization temperature', 'Crystallisation Temperature','Crystallization Temperature', 'Crystallisation temperature','Crystallization temperature',
                       'Tc', 'T c', 'T  c', 'T C', 'Tc1', 'T c1', 'T  c1', 'T C1', 'Tc2', 'T c2', 'T  c2', 'T C2', 'T c3', 'T C3', 'T  c3', 'Tc3', 'Tp', 'T p', 'T  p', 'T P', 'T cr', 'T p1', 'Tp1', 'T  p1', 'T P1', 'T p2', 'Tp2', 'T P2', 
                       'T  p2', 'T x1', 'Tx1', 'T  x1', 'T X1', 'T x2', 'Tx2', 'T  x2', 'T X2', 'T x/degC', 'Tx/degC', 'T c/degC', 'Tc/degC', 'T cryst', 'T o', 'T O', 'Crystallization temperature, T x', 'Crystallization temperature, T c', 'Crystallization initiation temperature', 'Crystallization peak temperature'],
'MeltingTemp':['T m','melting point', 'Melting Point', 'Melting point', 'Tm', 'T  m', 'T m/degC', 'T m1', 'T m2', 'T m3', 'Tm3', 'Tm1', 'T  m1', 'Tm2', 'T  m2', 'Melting temperature', 'Melting Temperature', 'Melting peak', 'Melting Peak', 'T melt', 'T Melt', 'T M'],
'ExpansionCoeff':['cte','tec', 'CTE','TEC','coefficient of thermal expansion','thermal expansion coefficient', 'Coefficient of Thermal Expansion','Thermal Expansion Coefficient', 'Coefficient of thermal expansion','Thermal expansion coefficient', 'Thermal coefficient, a', 'a', 'Thermal expansion'],
'RefractiveIndex':['refractive index', 'Refractive Index', 'Refractive index', 'Refractive index, n e', 'Refractive index, n', 'Refractive index, n D', 'Refractive index, n d', 'Refractive index, n E', 'Refractive index, ni', 'Refractive index, n i', 'Refractive index, nI',
                   'n', 'ni', 'nd','nf','nc','ne', 'nC','nD', 'nE', 'nF', 'n c', 'n d','n e', 'n f', 'n C', "n C'", 'n D', 'n E', 'n F', "n F'", 'n 0', 'n  at 632.8nm', 'n at 632.8nm', 'n at 633nm', 'n @ 633nm', 'n @ 632.8nm'],
'ShearModulus':['shear modulus', 'Shear Modulus', 'Shear modulus', 'G', 'Shear modulus G'],
'BulkModulus':['bulk modulus', 'Bulk Modulus', 'Bulk modulus', 'B', 'K', 'Bulk modulus K s', 'Bulk modulus K'],
'LiquidusTemperature':['liquidus temperature', 'Liquidus temperature', 'Liquidus Temperature', 'T l','T L', 'Liquidus T', 'T L,mea', 'T L,cal', 'T liq', 'T L,ls', 'T liquidus'],
'VickersHardness':['hardness', 'Hardness', 'Vh','Hv','microhardness',"Vicker's hardness", "Vicker's Hardness", "vicker's hardness", 'Vickers hardness', 'Vickers Hardness', 'vickers hardness', 'Hardness VHN', 'H V', 'H v', 'h v', 'Vikers hardness', 'Hardness H v', 'Hardness Hv'],
'YoungsModulus':["Young's modulus", "Young's Modulus", "young's modulus", 'Longitudinal modulus', 'Elastic modulus', 'Em',  'E m', 'E', 'C11', 'C22', 'C33', 'C12', 'C13', 'C23', 'C44', 'C 11', 'C 22', 'C 33', 'C 12', 'C 13', 'C 23', 'C 44', "Young's modulus E"],
'PoissonRatio':['Poisson', 'poisson', 'nu', 'Poisson ratio', 'Poisson Ratio', 'poisson ratio', 'v', "Poisson's ratio s", "Poisson's ratios", "Poisson's ratio", "Poisson's Ratio", "poisson's ratio", "Poisson's constant", "Poisson's Ratio, n", "Poisson's Ratio, s", 's', 'n'],
'DielectricConst':['Dk','dielectric constant', 'Dielectric Constant', 'Dielectric constant'],
'ElectricConduct':['electrical conductivity', 'Electrical Conductivity', 'Electrical conductivity', 'AC', 's', 's 0', 's -1', 's0', 'log s 298', 's 25 degC', 's25', 's 25', 's298', 's 298', 'conductivity', 'Conductivity'],
'AbbeValue':['abbe value','abbe number', 'Abbe Value','Abbe Number', 'Abbe value','Abbe number','Vd','Ve', 'v d', 'v e', 'Abbe no. n', 'Abbe no.', 'Abbe number, u D', 'v D', 'V D', 'VD'],
'TSofP':['softening point','softening temperature', 'Softening Point','Softening Temperature', 'Softening point','Softening temperature','Ts','T s', 'Softening point:'],
'TAnnealingP':['annealing point','annealing temperature', 'Annealing Point','Annealing Temperature', 'Annealing point','Annealing temperature', 'T 12', 'Annealing point:', 'T12', 'T ann', 'T an', 'Annealing temperature T'],
'FractureToughness':['fracture toughness', 'Fracture Toughness', 'Fracture toughness', 'K1C', 'Kc','KIC', 'k1c', 'kc','kic', 'K1c', 'K IC', 'K 1C', 'K Ic', 'K 1c', 'K C', 'K c', 'Fracture toughness, K IC', 'Fracture toughness, KIC', 'Toughness K c', 'Toughness Kc'],
'ActivationEnergy' :['E0', 'Ae', 'Ea', 'Activation Energy', 'Activation energy', 'activation energy', 'E a', 'Ec', 'E c', 'E A', 'E s', 'E dc', 'Edc', 'Es'],
'SofP' :['T d', 'T D', 'Td', 'T ds', 'T f', 'Tf', 'T F'],
}
#'DCBreakdownVoltage':['DC Breakdown Voltage', 'DC breakdown voltage']
#Activation energy: add substring rule: activation energy in heading

for keys in subs_dict.keys():
    if keys not in prop_names:
        print(keys)


In [13]:
new_train_dict = {key: train_dict[key] for key in keys_to_extract}
my_new_dict = {prop_names[i]: 0 for i in range(20)}

In [14]:
# def clean_names(column_names):
#     cleaned_names, unit_names = [], []
#     for name in column_names:
#         cleaned_name = re.sub(r'(\[.*\]|\(.*\)|\{.*\}|\<.*\>)([^a-zA-Z]*)', '', name).strip()
#         cleaned_names.append(cleaned_name)
#     return cleaned_names

In [15]:
def clean_names(column_names):
    cleaned_names, unit_names = [], []
    for name in column_names:
        match = re.search(r'(\[.*\]|\(.*\)|\{.*\}|\<.*\>)([^a-zA-Z]*)', name)
        if match:
            unit_name = match.group(1)
            cleaned_name = re.sub(r'(\[.*\]|\(.*\)|\{.*\}|\<.*\>)([^a-zA-Z]*)', '', name).strip()
            cleaned_names.append(cleaned_name)
            unit_names.append(unit_name[1:-1])
        else:
            cleaned_names.append(name)
            unit_names.append(None)  # If no match, you can choose how to handle this case
    return cleaned_names, unit_names

# Example usage:
# column_names = ["Example [Unit1]", "Another (Unit2)", "Test {Unit3}", "NoUnit", "E dc(TSPC) (kJ mol-1)"]
# cleaned, units = clean_names(column_names)
# print("Cleaned Names:", cleaned)
# print("Unit Names:", units)


In [16]:
def contains_at_least_three_numbers(lst):
    num_of_numbers = sum(
        isinstance(element, (int, float)) or 
        (isinstance(element, str) and (element.replace('.', '', 1).isdigit() or (element[:-1].isdigit() if element.endswith('K') else False)))
        for element in lst
    )
    return num_of_numbers >= 3

In [17]:
def find_num(string):
    #remove tabs or unnecessary spaces
    try:
        string = string.strip()
    except:
        pass
    #e.g. already in int or float form: 12.5 -> 12.5
    try:
        if string[0] == '-':
            return float(string[1:])*(-1)
        else:
            return float(string)
    except:
        pass
    # e.g. 99.1x10-7
    range_regex = re.compile('^\d+\.?\d*\s*x\s*\d+\.?\d*[-|+]?\s*\d+\.?\d*$')
    try:
#         print('alla')
        ranges = range_regex.search(string).group().split('x')
        if '-' in ranges[1]:
            numu = ranges[1].split('-')
#             print(ranges)
#             print(float(numu[0]), float(numu[1]))
            num = float(ranges[0]) * pow(float(numu[0]), (-float(numu[1])))
            formatted_result = "{:.3e}".format(num)
            return formatted_result
        elif '+' in ranges[1]:
            numu = ranges[1].split('+')
            num = float(ranges[0]) * pow(float(numu[0]), (float(numu[1])))
#             print(num)
            return num
        num = float(ranges[0]) * float(ranges[1])
        formatted_result = "{:.3e}".format(num)
        return formatted_result
    except:
        pass
    #e.g. 12.5 - 13.5 -> 12.5
    range_regex = re.compile('^\d+\.?\d*\s*-\s*\d+\.?\d*')
    try:
        ranges = range_regex.search(string).group().split('-')
        num = float(ranges[0])
#         print(num)
        return num
    except:
        pass
#     print('alla')
    #e.g. 12.2 (5.2) -> 12.2
    bracket_regex = re.compile('(\d+\.?\d*)\s*\(\d*.?\d*\)')
    try:
        extracted_value = float(bracket_regex.search(string).group(1))
        return float(extracted_value)
    except:
        pass
    #e.g. 12.3 ± 0.5 -> 12.3
    plusmin_regex = re.compile('^(\d+\.?\d*)(\s*[±+-]+\s*\d+\.?\d*)')
    try:
        extracted_value = float(plusmin_regex.search(string).group(1))
        return extracted_value
    except AttributeError:
        pass
    #e.g. <0.05 -> 0.05  |  >72.0 -> 72.0    | ~12 -> 12
    lessthan_roughly_regex = re.compile('([<]|[~]|[>])=?\s*\d+\.*\d*')
    try:
        extracted_value = lessthan_roughly_regex.search(string).group()
        num_regex = re.compile('\d+\.*\d*')
        extracted_value = num_regex.search(extracted_value).group()
        return float(extracted_value)
    except:
        pass
    # e.g. 0.4:0.6 (ratios)
    if ':' in string:
        split = string.split(":")
        try:
            extracted_value = round(float(split[0])/float(split[1]), 3)
            return extracted_value
        except:
            pass
    # e.g. 220 [29] --> 220.0, where citations given, rejecting ab 220 [29] or 7 220 [29]
    if '[' in string and ']' in string:
        try:
            extracted_value = string[:string.index('[')]
            return float(extracted_value)
        except:
            pass
    # e.g. 723 (first peak or other text) --> 723.0
    if '(' in string and ')' in string:
        try:
            extracted_value = string[:string.index('(')]
            return float(extracted_value)
        except:
            pass
    # e.g. '150K' or '150 degC' --> 150.0 but not '1350degC/2h' or 'njn 150 njn'
    try:
        exact_number_regex = re.compile('^(\d+\.?\d*)\s*[a-zA-Z]+$')
        # Using search to find the first occurrence of the pattern in the string
        match = exact_number_regex.search(string)
#         print('Match')
#         print(match)
        
        # If a match is found, extract the numeric part and convert to float
        if match:
            numeric_part = match.group(1)
            extracted_value = float(numeric_part)
            return extracted_value
    except:
        pass
    return None

In [18]:
# print(find_num('3.10x10-9'))
# ab  = '3.10x10-9	'
# # print(float(ab.strip()))

In [19]:
# check_piis = [('S0022309302010591', 0),
#  ('S002230930500654X', 0),
#  ('S0022309307007983', 1),
#  ('S0022309399006882', 0),
#  ('S0022309307005716', 0),
#  ('S0022309306009914', 0),
#  ('S0022309396006047', 1),
#  ('S0022309308002962', 0),
#  ('S0022309306005400', 0),
#  ('S0022309309000532', 0),
#  ('S0022309398004335', 0),
#  ('S0022309315301307', 1),
#  ('S0022309303006306', 0),
#  ('S0022309302010530', 1),
#  ('S0038109807002189', 0),
#  ('S0022309300004087', 0),
#  ('S0022309397000148', 3),
#  ('S0022309315001556', 0),
#  ('S0022309307003109', 1),
#  ('S0022309399006018', 1),
#  ('S0022309398008096', 0),
#  ('S0022309307006230', 0),
#  ('S0022309304006003', 0)]

In [20]:
def check_exception(prop_name, row_col_elem, element, unit, caption):
    
    if element == 'r' or element == 'd' or element == 'D' and prop_name == 'Density':
        if unit == None: #same as unit is None
            return False
        arr = ['/cm3', 'cm-3', '/ cm3', 'cm -3', '/cc', '/cm 3', 'cm -3', '/ cc', 'cc-1', 'cc -1', '/m3', '/m 3', '/ m3','m-3', 'm -3']
        found = any([element in unit for element in arr]) and 'g' in unit
        return found
    
    if element in ['E', 'G', 'K', 'B', 'Em', 'E m', 'C11', 'C22', 'C12', 'C13', 'C23']:
        if unit == 'GPa':
            return True
        else:
            return False
        
    if element== 'a':
        if unit == None: #same as unit is None
            return False
        arb = ['/degC', 'degC-1', '/K', 'K-1', 'K -1', 'degC -1']
        foundd = any([element in unit for element in arb])
        return foundd
    
    if element in ['Tc', 'T c', 'T  c', 'T C']:
        cap_ch = 'Curie temperature'
        cap2_ch = 'Temperatures of Curie'
        if cap_ch.lower() in caption.lower() or cap2_ch.lower() in caption.lower():
            return False
        
    if element in ['Tm', 'Tm1', 'Tm2', 'Tm3', 'T M']:
        cap_ch = 'melting '
        if cap_ch.lower() in caption.lower():
            return True
        else:
            return False
        
    if element in ['T f', 'Tf', 'T F']:
        cap_ch = 'softening '
        if cap_ch.lower() in caption.lower():
            return True
        else:
            return False
        
    if element in ['AC', 's', 's 0', 's -1', 's0', 'log s 298', 's 25 degC', 's25', 's 25', 's298', 's 298', 'conductivity', 'Conductivity'] and prop_name=='ElectricConduct':
        arb = ['O-1 cm- 1', 'O-1 cm-1', 'ohm-1 cm-1', 'Scm-1', 'S cm-1', 'S/cm', 'O-1 m- 1', 'O-1 m-1', 'ohm-1 m-1', 'Sm-1', 'S m-1', 'S/m']
        if unit == None: #same as unit is None
            foundd = False
        else:  
            foundd = any([element in unit for element in arb])
        cap_ch = ['electrical ', 'electric ', 'room', 'ac ', 'dc ', 'ionic'] 
        cap_ch_1 = 'conductivit'
        cap_check = False
        if any([word in caption.lower() for word in cap_ch]) and cap_ch_1 in caption.lower():
            cap_check =  True
        else:
            cap_check = False
        return foundd or cap_check
    
    if element in ['v']:
        all_elem_less_1 = all((0 < float(find_num(elem)) < 1) if find_num(elem)!=None else True for elem in row_col_elem)
        return all_elem_less_1
    
    if element in ['E0', 'Ae', 'Ea', 'E a', 'Ec', 'E c', 'E A', 'E s', 'E dc', 'Edc', 'Es'] and prop_name == 'ActivationEnergy':
        if unit == None: #same as unit is None
            foundd = False
        else:    
            arb = ['/mol', '/ mol', 'mol-1', 'mol -1', '/at', '/ at', 'at-1', 'at -1', 'eV']
            foundd = any([element in unit for element in arb])
        #cap_ch = 'energy band gap'
        cap_ch2 = 'activation ene'
        cap_check = False
#         if cap_ch.lower() in caption.lower():
#             if cap_ch2.lower() in caption.lower():
#                 cap_check = True
#             else:
#                 cap_check = False
        if cap_ch2.lower() in caption.lower():
            cap_check = True
        return foundd or cap_check
    
    if element in ['s', 'n'] and prop_name == 'PoissonRatio':
        #check no conductivity in caption but Poisson in caption for s
        #check no refractive in caption but Poisson in caption for n
        #range of value should be btwn [0.1, 0.5]
        cap_ch = 'poisson'
        cap_ch1 = 'conductivity'
        cap_ch2 = 'refractive'
        cap_check = False
        
        if element == 's':
            if cap_ch in caption.lower() and cap_ch1 not in caption.lower():
                cap_check = True
        elif element == 'n':
            if cap_ch in caption.lower() and cap_ch2 not in caption.lower():
                cap_check = True
        
        all_elem_in_range = all((-1 <= float(find_num(elem)) <= 0.5) if find_num(elem) is not None else True for elem in row_col_elem)
        return cap_check and all_elem_in_range
        
        
#     if element in ['Ec', 'E c']:
#         cap_ch = 'energy band gap'
#         cap_ch2 = 'activation ene'
#         cap_check = True
#         if cap_ch.lower() in caption.lower():
#             if cap_ch2.lower() in caption.lower():
#                 cap_check = True
#             else:
#                 cap_check = False
#         return cap_check
    return True

In [21]:
def check_exception_ri(prop_name, unit, row_col_elem, element, caption, pii_tidx):
    if prop_name != 'RefractiveIndex':
        return False
    row_col_elem = row_col_elem[1:]
    all_elem_less_10, unit_check, caption_check, len_check = False, False, False, False
    #all_elem_less_10 = all([0<elem<10 if isinstance(elem, (int, float)) else True for elem in row_col_elem]) and any([isinstance(eleme, (int, float)) for eleme in row_col_elem])
    #all_elem_less_10 = all((1 < float(find_num(elem)) < 10) if elem.replace('.', '', 1).isdigit() else True for elem in row_col_elem) and any(elem.replace('.', '', 1).isdigit() for elem in row_col_elem)
    all_elem_less_10 = all((1 < float(find_num(elem)) < 10) if find_num(elem)!=None else True for elem in row_col_elem)
    if unit!=None:
        unit_check = 'nm' in unit or 'mm' in unit
    else:
        unit_check = True
    if len(element)>2:
        len_check = True
    if caption!=None:
        caption_check = 'refractiv' in caption.lower() or 'optical prop' in caption.lower()
        #caption_check = 'Refractive' in caption or 'refractive' in caption or 'REFRACTIVE' in caption
    #print(pii_tidx)
#     if pii_tidx == ('S0022309302017982', 0):
#         print('HHEeeeeeenrnrjirjrjrork-----')
#         print(row_col_elem)
#         print(unit, caption, pii_tidx)
#         print(all_elem_less_10, unit_check, caption_check, len_check)
#         print()
    conditions = [unit_check, caption_check, len_check]
    if sum(conditions) >= 2:
        cond_result = True
    else:
        cond_result = False
    return all_elem_less_10 and cond_result

In [22]:
# # row_col_elem = ['abc',3.5, 6.7, 2.3]
# row_col_elem = ['n', '2.29', '2.28', '2.27', '2.26']
# up_arr = all((0 < float(elem) < 10) if elem.replace('.', '', 1).isdigit() else True for elem in row_col_elem) and any(elem.replace('.', '', 1).isdigit() for elem in row_col_elem)
# up_arr
#prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'SofP', 'BulkModulus', 'ActivationEnergy']


In [23]:
def direct_matching(element, prop_name):
#     if ('density' in element.lower()) and (prop_name == 'Density'): # relative density, atomic density, power density, energy density, current density
#         return prop_names.index(prop_name) + 4

    if ('transition temperature' in element.lower()) and (prop_name == 'GlassTransitionTg'):
        return prop_names.index(prop_name) + 4

    elif ('refractive index' in element.lower() and 'non' not in element.lower()) and (prop_name == 'RefractiveIndex'):
        return prop_names.index(prop_name) + 4

    elif ('abbe number' in element.lower() or 'abbe value' in element.lower()) and prop_name == 'AbbeValue':
        return prop_names.index(prop_name) + 4

    elif ('young' in element.lower() and 'modulus' in element.lower()) and (prop_name == 'YoungsModulus'):
        return prop_names.index(prop_name) + 4

    elif ('shear modulus' in element.lower()) and (prop_name == 'ShearModulus'):
        return prop_names.index(prop_name) + 4

    elif ('hardness ' in element.lower()) and (prop_name == 'VickersHardness'):
        return prop_names.index(prop_name) + 4

    elif (('poisson' in element.lower() and 'ratio' in element.lower())) and (prop_name == 'PoissonRatio'):
        return prop_names.index(prop_name) + 4

    elif ('temperature' in element.lower() and 'crystallization' in element.lower()) and (prop_name == 'CrystallizationTemp'):
        return prop_names.index(prop_name) + 4

    elif ('dc conductiv' in element.lower() or 'electrical conductiv' in element.lower()) and (prop_name == 'ElectricConduct'):
        return prop_names.index(prop_name) + 4

    elif ('dielectric constant' in element.lower()) and (prop_name == 'DielectricConst'):
        return prop_names.index(prop_name) + 4

    elif ('annealing temperature' in element.lower() or 'annealing point' in element.lower()) and (prop_name == 'TAnnealingP'):
        return prop_names.index(prop_name) + 4

    elif ('thermal expansion coefficient' in element.lower() or 'coefficient of thermal expansion' in element.lower()) and (prop_name == 'ExpansionCoeff'):
        return prop_names.index(prop_name) + 4

    elif ('liquidus temp' in element.lower()) and (prop_name == 'LiquidusTemperature'):
#         print(element)
        return prop_names.index(prop_name) + 4

    elif ('melting temp' in element.lower()) and (prop_name == 'MeltingTemp'):
#         print(element)
        return prop_names.index(prop_name) + 4

    elif 'softening point' in element.lower() or 'softening temp' in element.lower() and prop_name == 'SofP':
        return prop_names.index(prop_name) + 4
#                 
    elif ('bulk modulus' in element.lower()) and (prop_name == 'BulkModulus'):
        return prop_names.index(prop_name) + 4

    elif ('activation energy' in element.lower()) and (prop_name == 'ActivationEnergy'):
        return prop_names.index(prop_name) + 4

    else: 
        return -1
                  

In [24]:
upd_table_list = []
tc_check_list = []
for index, key_table in enumerate(keys_to_extract):
    table = new_train_dict[key_table]
    pii_tidx = (table['pii'], table['t_idx'])
    caption = table['caption']
    un_first_row = table['act_table'][0]
    un_first_col = [row[0] for row in table['act_table']]
    first_row, units_frow = clean_names(un_first_row)
    first_col, units_fcol = clean_names(un_first_col)
    ri_check_list = ['nd','nf','nc','ne', 'nC','nD', 'nE', 'nF', 'n c', 'n d','n e', 'n f', 'n C', 'n D', 'n E', 'n F', 'n']
    if 'row_label' not in table.keys() or 'col_label' not in table.keys():
        table['row_label'] = [0] * table['num_rows']
        table['col_label'] = [0] * table['num_cols']
#     print(table)
#     print(first_row)
#     print(first_col)
#     if index == 1:
#         break
    
    for ind, element in enumerate(first_row):
        col_elem = [row[ind] for row in table['act_table']]
        for prop_name in subs_dict.keys():
            prop_name_list = subs_dict[prop_name]
#             if pii_tidx == ('S002230931000030X', 0):
#                 print('eelementt')
#                 print(element)
#             if element in ['g']:
#                 tc_check_list.append(pii_tidx)
#             if element not in prop_name_list and 'refractive index' in element and prop_name=='RefractiveIndex':
#                 print(pii_tidx)
#                 print(element)
#                 print(prop_name)
#                 print()
#             if pii_tidx == ('S002230931000030X', 0) and element == 's' and prop_name == 'PoissonRatio':
#                     print('Still here')
#                     print(element)
#                     print(element in prop_name_list)
#                     print(check_exception(prop_name,col_elem, element, units_frow[ind], caption))
            
            if direct_matching(element, prop_name) != -1:
                table['col_label'][ind] = direct_matching(element, prop_name)
                continue
                
            if element in prop_name_list and check_exception(prop_name,col_elem, element, units_frow[ind], caption):
            #if element in prop_name_list and check_exception(element, units_frow[ind]):
                if element in ri_check_list:
#                     col_elem = [row[ind] for row in table['act_table']]
#                     if table['pii'] == 'S0038109805003212' and element == 'n d':
#                         print('Founddd')
#                         print(pii_tidx)
#                         print(prop_name)
#                         print(prop_name_list)
#                         display(pd.DataFrame(table['act_table']))
#                         print('--------------------------------')
#                         print(check_exception_ri(units_frow[ind], col_elem, caption, pii_tidx))
                    if not check_exception_ri(prop_name, units_frow[ind], col_elem, element, caption, pii_tidx):
                        continue
#                 if table['pii'] == 'S0022309302017982' and element == 'n 0':
#                     print(table)
#                     print(ind)
#                     print(first_row)
#                     print('---------Controversy --------')
                table['col_label'][ind] = prop_names.index(prop_name) + 4

    for ind, element in enumerate(first_col):
        if ind==0: continue
        row_elem = table['act_table'][ind]
        for prop_name in subs_dict.keys():
            prop_name_list = subs_dict[prop_name]
#             if element in ['g']:
#                 tc_check_list.append(pii_tidx)

            if direct_matching(element, prop_name) != -1:
                table['row_label'][ind] = direct_matching(element, prop_name)
                continue
                
            if element in prop_name_list and check_exception(prop_name, row_elem, element, units_fcol[ind], caption):
                if element in ri_check_list:
#                     row_elem = table['act_table'][ind]
                    if not check_exception_ri(prop_name, units_fcol[ind], row_elem, element, caption, pii_tidx):
                        continue
              
#                 if table['pii'] == 'S0038109805003212':
#                     print(table)
#                     print(ind)
#                     print(first_col)
                table['row_label'][ind] = prop_names.index(prop_name) + 4
    
    if any(x >= 4 for x in table['row_label']) and any(x >= 4 for x in table['col_label']):
        check_row = contains_at_least_three_numbers(first_row)
        check_col = contains_at_least_three_numbers(first_col)
        if check_row and not check_col:
            table['col_label'] = [0 if element >= 4 else element for element in table['col_label']]
        elif not check_row and check_col:
            table['row_label'] = [0 if element >= 4 else element for element in table['row_label']]
        else:
            count_col = sum(1 for element in table['col_label'] if element >= 4)
            count_row = sum(1 for element in table['row_label'] if element >= 4)
            if count_col>= count_row:
                table['row_label'] = [0 if element >= 4 else element for element in table['row_label']]
            else:
                table['col_label'] = [0 if element >= 4 else element for element in table['col_label']]
#         print(table['pii'], table['t_idx'])
#     if (table['pii'], table['t_idx']) in check_piis:
#         #print(table)
#         print(first_row)
#         print(table['col_label'])
#         print(first_col)
#         print(table['row_label'])
#         print()
    
    upd_table_list.append(table)

In [25]:
tc_check_list = list(set(tc_check_list))
for elem in tc_check_list:
    print(f'https://www.sciencedirect.com/science/article/pii/{elem[0]}#TBL{elem[1]+1}')

In [26]:
print(len(upd_table_list))

3983


In [27]:
my_upd_dict = {prop_names[i]: 0 for i in range(20)}
pii_tid_str = {prop_names[i]: set() for i in range(20)}
all_rows, all_cols = [], []
pii_tidx_ds = set()
for table in upd_table_list:
#     if table['pii'] == 'S0022309304004089' and table['t_idx'] == 0:
#         print(table['row_label'])
#         display(pd.DataFrame(table['act_table']))
    all_rows += table['row_label']
    all_cols += table['col_label']
    pii_tidx_ds.add((table['pii'], table['t_idx']))
    tab_row_col = table['row_label'] + table['col_label']
    for ind, elem in enumerate(tab_row_col):
        if elem>=4:
            pii_tid_str[prop_names[elem-4]].add((table['pii'], table['t_idx'], ind))
for keys in range(4, 24):
    my_upd_dict[prop_names[keys-4]] = all_rows.count(keys) + all_cols.count(keys)

In [28]:
pickle.dump(pii_tid_str, open(os.path.join(table_dir, 'check_precision_string_matching.pkl'), 'wb'))

In [29]:
from collections import defaultdict
total_dict = defaultdict(set)
count_con = 0
remove_prop = []
for key in my_upd_dict.keys():
    total_dict[key] = my_dict[key] + my_upd_dict[key]
    if total_dict[key]>=5:
        count_con += 1
    else:
        remove_prop.append(key)
print(total_dict)
print(count_con)
print(len(remove_prop))
print(remove_prop)

defaultdict(<class 'set'>, {'Density': 352, 'GlassTransitionTg': 483, 'RefractiveIndex': 167, 'AbbeValue': 10, 'YoungsModulus': 62, 'ShearModulus': 17, 'VickersHardness': 47, 'PoissonRatio': 29, 'FractureToughness': 12, 'CrystallizationTemp': 385, 'MeltingTemp': 94, 'ElectricConduct': 48, 'DielectricConst': 13, 'TSofP': 16, 'TAnnealingP': 32, 'ExpansionCoeff': 61, 'LiquidusTemperature': 75, 'SofP': 18, 'BulkModulus': 20, 'ActivationEnergy': 102})
20
0
[]


In [30]:
final_upd_table_list = []
for table in upd_table_list:
    row_col_labels = table['row_label'] + table['col_label']
    if any(elem>=4 for elem in row_col_labels):
        final_upd_table_list.append(table)
# print(len(final_upd_table_list))

final_upd_table_list = []
for table in upd_table_list:
    row_col_labels = table['row_label'] + table['col_label']
    if any(elem>=4 for elem in row_col_labels):
        final_upd_table_list.append(table)
# print(len(final_upd_table_list))

In [31]:
pickle.dump(final_upd_table_list, open(os.path.join(table_dir, 'all_string_match_data_pre.pkl'), 'wb'))

In [32]:
print(len(final_upd_table_list))

596
